In [7]:
!pip install transformers datasets accelerate gdown

In [8]:
#!gdown 1kJXtUko-70rNP250KxkiW3OTE7bg1RAo

Downloading...
From: https://drive.google.com/uc?id=1kJXtUko-70rNP250KxkiW3OTE7bg1RAo
To: /home/jovyan/luna16_OG/final_dataset.csv
100%|██████████████████████████████████████| 64.7M/64.7M [00:02<00:00, 27.6MB/s]


In [10]:
import numpy as np
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [11]:
df = pd.read_csv("final_dataset.csv")

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.metrics import classification_report, f1_score

# ---------------------------------------------------------
# 1) Ucitavanje i priprema podataka
# ---------------------------------------------------------
df = pd.read_csv("final_dataset.csv")

# baciti redove bez teksta (nema sta trenirati)
df = df.dropna(subset=["SADRZAJ"]).reset_index(drop=True)

TOPICS = [
    "euroatlantske_integracije",
    "negiranje_genocida",
    "gradjanska_vs_konstitutivni",
    "izborna_reforma",
]

def make_label(row, topic):
    """mentioned + stance -> not_mentioned / against / neutral / for"""
    if not row[f"{topic}_mentioned"]:
        return "not_mentioned"
    stance = row[f"{topic}_stance"]
    if stance == 1:
        return "for"
    elif stance == -1:
        return "against"
    else:
        return "neutral"

for topic in TOPICS:
    df[f"{topic}_label"] = df.apply(lambda r: make_label(r, topic), axis=1)

# ---------------------------------------------------------
# 2) TF-IDF (word + char_wb n-grami) preko FeatureUnion
# ---------------------------------------------------------
def build_vectorizer():
    return FeatureUnion([
        ("word", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.9,
            sublinear_tf=True,
        )),
        ("char", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_df=0.9,
            sublinear_tf=True,
        )),
    ])

In [18]:
df["TEXT_BERT"] = (
    df["naslov"].fillna("") + ". " +
    df["SADRZAJ"].fillna("")
)

In [33]:
from transformers import EarlyStoppingCallback

def train_bert_for_topic(topic, model_name="classla/bcms-bertic"):
    print("=" * 70)
    print(f"TEMA: {topic} | BERT FINE-TUNING")
    print("=" * 70)

    labels = ["against", "for", "neutral", "not_mentioned"]
    label2id = {label: i for i, label in enumerate(labels)}
    id2label = {i: label for label, i in label2id.items()}

    temp = df[["TEXT_BERT", f"{topic}_label"]].dropna().copy()
    temp["label"] = temp[f"{topic}_label"].map(label2id)

    train_df, test_df = train_test_split(
        temp,
        test_size=0.2,
        random_state=42,
        stratify=temp["label"]
    )

    train_ds = Dataset.from_pandas(train_df[["TEXT_BERT", "label"]])
    test_ds = Dataset.from_pandas(test_df[["TEXT_BERT", "label"]])

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(
            batch["TEXT_BERT"],
            truncation=True,
            padding="max_length",
            max_length=512
        )

    train_ds = train_ds.map(tokenize, batched=True)
    test_ds = test_ds.map(tokenize, batched=True)

    train_ds = train_ds.remove_columns(["TEXT_BERT", "__index_level_0__"])
    test_ds = test_ds.remove_columns(["TEXT_BERT", "__index_level_0__"])

    train_ds.set_format("torch")
    test_ds.set_format("torch")

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(labels),
        id2label=id2label,
        label2id=label2id
    )

    def compute_metrics(eval_pred):
        logits, y_true = eval_pred
        y_pred = np.argmax(logits, axis=1)

        return {
            "accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        }

    args = TrainingArguments(
        output_dir=f"./bert_results_{topic}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=10,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=50,
        save_total_limit=1,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]

    )

    trainer.train()

    pred = trainer.predict(test_ds)
    y_pred = np.argmax(pred.predictions, axis=1)
    y_true = pred.label_ids

    print(classification_report(
        y_true,
        y_pred,
        target_names=labels,
        zero_division=0
    ))

    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    return {
        "trainer": trainer,
        "model": model,
        "tokenizer": tokenizer,
        "macro_f1": macro_f1,
        "label2id": label2id,
        "id2label": id2label,
    }

In [34]:
bert_result_euro = train_bert_for_topic("euroatlantske_integracije")

TEMA: euroatlantske_integracije | BERT FINE-TUNING


Map:   0%|          | 0/12004 [00:00<?, ? examples/s]

Map:   0%|          | 0/3002 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: classla/bcms-bertic
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

               precision    recall  f1-score   support

      against       0.58      0.60      0.59       249
          for       0.83      0.56      0.67       271
      neutral       0.47      0.51      0.49       308
not_mentioned       0.92      0.94      0.93      2174

     accuracy                           0.83      3002
    macro avg       0.70      0.65      0.67      3002
 weighted avg       0.83      0.83      0.83      3002



In [35]:
bert_result_gradanska = train_bert_for_topic("gradjanska_vs_konstitutivni")

TEMA: gradjanska_vs_konstitutivni | BERT FINE-TUNING


Map:   0%|          | 0/12004 [00:00<?, ? examples/s]

Map:   0%|          | 0/3002 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: classla/bcms-bertic
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

               precision    recall  f1-score   support

      against       0.71      0.67      0.69       415
          for       0.68      0.47      0.56       166
      neutral       0.19      0.25      0.22        48
not_mentioned       0.94      0.96      0.95      2373

     accuracy                           0.88      3002
    macro avg       0.63      0.59      0.60      3002
 weighted avg       0.88      0.88      0.88      3002



In [36]:
bert_result_izborna = train_bert_for_topic("izborna_reforma")

TEMA: izborna_reforma | BERT FINE-TUNING


Map:   0%|          | 0/12004 [00:00<?, ? examples/s]

Map:   0%|          | 0/3002 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: classla/bcms-bertic
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

               precision    recall  f1-score   support

      against       0.63      0.46      0.53       129
          for       0.55      0.38      0.44        48
      neutral       0.61      0.38      0.47        82
not_mentioned       0.96      0.99      0.98      2743

     accuracy                           0.94      3002
    macro avg       0.69      0.55      0.61      3002
 weighted avg       0.93      0.94      0.94      3002



In [37]:
bert_result_genocid = train_bert_for_topic("negiranje_genocida")

TEMA: negiranje_genocida | BERT FINE-TUNING


Map:   0%|          | 0/12004 [00:00<?, ? examples/s]

Map:   0%|          | 0/3002 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: classla/bcms-bertic
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

               precision    recall  f1-score   support

      against       0.53      0.57      0.55       127
          for       0.73      0.65      0.69       239
      neutral       0.30      0.42      0.35        48
not_mentioned       0.97      0.97      0.97      2588

     accuracy                           0.92      3002
    macro avg       0.63      0.65      0.64      3002
 weighted avg       0.92      0.92      0.92      3002

